In [5]:
import joblib, numpy as np, tensorflow as tf

model = tf.keras.models.load_model("saved_model/ae_model.keras")
scaler = joblib.load("saved_model/scaler.joblib")
meta = joblib.load("saved_model/detector_meta.joblib")

X = np.load("../data/processed/X_all.npy")
C = np.load("../data/processed/C_all.npy")

X_clean = X[:5000]
C_clean = C[:5000]

def apply_scaler(sc, X):
    N,T,F = X.shape
    return sc.transform(X.reshape(N*T, F)).reshape(N,T,F).astype(np.float32)

X_clean = apply_scaler(scaler, X_clean)

ValueError: File not found: filepath=../../saved_model/ae_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
def inject_anomalies(x, anomaly_ratio=0.08, seed=42, spike_mag=6.0, noise_std=0.8, seg_len=12, n_affect=2):
    rng = np.random.default_rng(seed)
    x_anom = x.copy()
    n, T, F = x_anom.shape

    k = int(n * anomaly_ratio)
    idx = rng.choice(n, k, replace=False)
    y = np.zeros(n, dtype=np.int32)
    y[idx] = 1

    for i in idx:
        seg = min(seg_len, T)
        t0 = int(rng.integers(0, max(1, T - seg + 1)))
        feats = rng.choice(F, size=min(n_affect, F), replace=False)
        t_idx = np.arange(t0, t0 + seg)

        mode = rng.choice(["spike","drop","noise"])
        if mode == "spike":
            x_anom[i][np.ix_(t_idx, feats)] += spike_mag
        elif mode == "drop":
            x_anom[i][np.ix_(t_idx, feats)] -= spike_mag
        else:
            x_anom[i][np.ix_(t_idx, feats)] += rng.normal(0, noise_std, size=(len(t_idx), len(feats)))

    return x_anom, y

X_test, y_test = inject_anomalies(X_clean, anomaly_ratio=0.08, seed=42)

In [2]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score

X_pred = model.predict([X_test, C_clean], verbose=0)
scores = topk_score(X_test, X_pred, meta["topk"])
yhat = (scores >= meta["threshold"]).astype(int)

p,r,f1,_ = precision_recall_fscore_support(y_test, yhat, average="binary", zero_division=0)
cm = confusion_matrix(y_test, yhat)

print("Precision:", p)
print("Recall   :", r)
print("F1       :", f1)
print("CM:\n", cm)
print("PR-AUC:", average_precision_score(y_test, scores))
print("ROC-AUC:", roc_auc_score(y_test, scores))

NameError: name 'model' is not defined